In [ ]:
from requests.adapters import HTTPAdapter 
from urllib3.util.retry import Retry
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

In [ ]:
all_listings = []
base_url = 'https://nigeriapropertycentre.com/for-rent/abuja?page='

#IMPROVEMENT 1
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.107 Safari/537.36"
]

#IMPROVEMENT 2
session = requests.Session()
retries = Retry(total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))
session.headers.update({"User-Agent": random.choice(user_agents)})

for page in range(1, 235):
    print(f"Scraping page {page}...") # Added print to track progress
    url = f"{base_url}{page}"
    
    try:
        #IMPROVEMENT 3
        headers = {"Referer": base_url + str(page - 1) if page > 1 else "https://nigeriapropertycentre.com"}
        
        response = session.get(url, headers=headers, timeout=15)
        
        
        if response.status_code != 200:
            print(f"Warning: Failed to load page {page}. Status code: {response.status_code}")
            continue

        soup = BeautifulSoup(response.text, 'html.parser')
        listings = soup.select(".wp-block-body")
        
        
        if not listings:
            print("No more listings found.")
            break

        for house in listings:
            try: #IMPROVEMENT 4
                prop_type = house.select(".content-title")
                property_type = [i.text.strip() for i in prop_type]

                init_price = house.select(".price")
                
                if len(init_price) >= 2:
                    currency = init_price[0].text
                    amount = init_price[1].text
                else:
                    currency = ""
                    amount = "0"

                address = house.select(".voffset-bottom-10")
                location = [i.text.strip() for i in address]

                listing_info = {
                    "Property Type": property_type[0] if property_type else "N/A",
                    "Price (Per Annum)": f"{currency}{amount}",
                    "Location": location[0] if location else "N/A"
                }
                all_listings.append(listing_info)
            except Exception as e:
                print(f"Error parsing a listing: {e}") 
                continue

        #IMPROVEMENT 5
        time.sleep(random.uniform(3, 7))

        #IMPROVEMENT 6
        if page % 10 == 0:
            df = pd.DataFrame(all_listings)
            df.to_csv(r"C:\Users\abba\Desktop\DS Projects\AMAC-Rental-Intelligence-Dashboard\Data\raw\npc_raw_listings.csv", index=False)
            print(f"checkpoint: saved data up to page {page}")

    except Exception as e:
        print(f"Critical error on page {page}: {e}")
        time.sleep(10) # Wait longer if a network error occurs


df = pd.DataFrame(all_listings)
df.to_csv(r"C:\Users\abba\Desktop\DS Projects\AMAC-Rental-Intelligence-Dashboard\Data\raw\npc_raw_listings.csv", index=False)


In [ ]:
df.shape